In [2]:
import pandas as pd
import os
from pathlib import Path

In [3]:
# Vérifions que le répertoire Direction Risques existe

base = Path("/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0")
print("Existe ?", base.exists())
if base.exists():
    for p in sorted(base.iterdir()):
        print("-", p.name)
else:
    print("Dossier introuvable : vérifie le montage et le chemin.")


Existe ? True
- BP_SIM_EPS.json
- BT_SIM_EPS.json
- INDICATEURS_RISQUE.json
- PARAM_RRF.json
- PASSIF_SIM_EPS.json
- PRICING.json
- VA_SIM_EPS.json
- bp_eps.sas7bdat
- bp_sim_eps.sas7bdat
- bt_eps.sas7bdat
- bt_sim_eps.sas7bdat
- data_gse.sas7bdat
- flux_passif_sim.sas7bdat
- histo_indic.sas7bdat
- indicateurs_risque.sas7bdat
- param_rrf.sas7bdat
- passif_sim_eps.sas7bdat
- pricing.sas7bdat
- va_flux_eps.sas7bdat
- va_sim_eps.sas7bdat


In [4]:
base_transpa_sim_path = "/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0/BT_SIM_EPS.json"

#Bureautique\\Direction des Risques\\4. Risques Financiers\\00-REPORTING\\01 - HISTO SAS\\0000T0
# vérifion d'abord que le fichie BASE_TRANSPA existe
p = Path(base_transpa_sim_path)
print("Fichier existe ?", p.exists())

if p.exists():
    df_base_transpa = pd.read_json(p)
    print(df_base_transpa.head())
else:
    raise FileNotFoundError(f"Fichier introuvable: {base_transpa_sim_path}")

# Excluons toute les lignes ayant pour ID "TRANSFERT"
df_base_transpa = df_base_transpa[df_base_transpa["ID"] != "TRANSFERT"]

# Déterminons le type de données de nos colonnes

# Dates
for col in ['DATE_VALEUR','DATE_TRANSPA','ECHEANCE']:
    df_base_transpa[col] = pd.to_datetime(df_base_transpa[col], dayfirst=True, errors='coerce')

# Catégories
cols_cat = [
    'CODE_NIV0','CODE_NIV1','CODE_NIV2','CODE_NIV3','CODE_NIV4',
    'ID','DEVISE','SOURCE','STATUT_COURS','ATTRIBUT_W','CIC','CIC_EPS',
    'ID_EMETTEUR','LIBELLE','PAYS_ISO','PAYS','CLASSIF_RF','CLASSIF_RISQUE_ALM',
    'CLASSIF_SAA','CODE_CIC_ALM','COTATION','LIB_CIC_EPS','CPN_TYPE',
    'INDIC_CONV','INDIC_COVD','INDIC_INDEX','INDIC_SUBD','INDIC_ZC',
    'REFERENCE_INDEX','STATUT_COURS_CC_EPS','LIB_EMETTEUR','TYPE_EMETTEUR',
    'QS_AGENCE','QS_AGENCE_EM','SECTEUR_EPS','SOUS_SECTEUR_EPS',
    'ID_GROUPE','LIB_GROUPE','TYPE_GROUPE','CODE_NACE'
]
for col in cols_cat:
    df_base_transpa[col] = df_base_transpa[col].astype('category')

# Booléens (si vraiment binaire, à vérifier dans les données)
for col in ['INDIC_CONV','INDIC_COVD','INDIC_INDEX','INDIC_SUBD','INDIC_ZC']:
    df_base_transpa[col] = df_base_transpa[col].map({'O': True, 'N': False}).astype('boolean')

for col in ['EEE','EM','OCDE','UE','ZE']:
    df_base_transpa[col] = df_base_transpa[col].map({'1': True, '0': False}).astype('boolean')

# Entiers petits (optimisation mémoire)
cols_smallint = ['NIV_TRANSPA','QUALITY_STEP','QUALITY_STEP_EM','RECLASSEMENT','CPN_FREQ']
for col in cols_smallint:
    df_base_transpa[col] = df_base_transpa[col].astype('Int8') 

Fichier existe ? True
  DATE_TRANSPA DATE_VALEUR  SOURCE GROUPE  CANTON AFFECTATION SECTION  \
0    31DEC2024   31DEC2024  CHORUS    CGP  CGP RS          RS           
1    31DEC2024   31DEC2024  CHORUS    CGP  CGP AG         MDD           
2    31DEC2024   31DEC2024  CHORUS    CGP  CGP RS          RS           
3    31DEC2024   31DEC2024  CHORUS    CGP  CGP RS          RS           
4    31DEC2024   31DEC2024  CHORUS    CGP  CGP AG          FP           

  CODE_CTN  CODE_PTF LIB_PTF  ...     CODE_NIV4                LIBELLE_P  \
0               5001          ...  AT0000383864  AUTRICHE 6,25%97-27 S.6   
1               2006          ...  AT0000A04967  AUTRICHE 4,15%07-150337   
2               5001          ...  AT0000A04967  AUTRICHE 4,15%07-150337   
3               5001          ...  AT0000A0U299  AUTRICHE 3,80%12-260162   
4               3001          ...  AT0000A0VRQ6  AUTRICHE 3,15%12-200644   

  ATTRIBUT_W_P              LIBELLE_W_P OPC_P         ID_EMETTEUR_P  \
0          

/tmp/ipykernel_817281/3772615920.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_base_transpa[col] = pd.to_datetime(df_base_transpa[col], dayfirst=True, errors='coerce')
/tmp/ipykernel_817281/3772615920.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_base_transpa[col] = pd.to_datetime(df_base_transpa[col], dayfirst=True, errors='coerce')
/tmp/ipykernel_817281/3772615920.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_base_transpa[col] = pd.to_datetime(df_base_transpa[col], dayfirst=True, errors='coerce')


In [5]:
# Sauvegarde des données brutes de Base Transpa
# On s'assure que le dossier cleaned existe
os.makedirs("./raw", exist_ok=True)

# Sauvegarde de df_base_transpa en CSV
output_path = "./raw/raw_base_transpa.csv"
df_base_transpa.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./raw/raw_base_transpa.csv
Taille du fichier :  49.26 Mo


In [6]:
df_base_transpa.info(verbose= True)

<class 'pandas.DataFrame'>
RangeIndex: 50026 entries, 0 to 50025
Data columns (total 125 columns):
 #    Column                Dtype         
---   ------                -----         
 0    DATE_TRANSPA          datetime64[us]
 1    DATE_VALEUR           datetime64[us]
 2    SOURCE                category      
 3    GROUPE                str           
 4    CANTON                str           
 5    AFFECTATION           str           
 6    SECTION               str           
 7    CODE_CTN              str           
 8    CODE_PTF              int64         
 9    LIB_PTF               str           
 10   TYPE_GESTION          str           
 11   TYPE_GESTION_2        int64         
 12   ID                    category      
 13   LIBELLE               category      
 14   DEVISE                category      
 15   STATUT_COURS          category      
 16   QUANTITE              float64       
 17   QUANTITE_IDX          float64       
 18   VB_CC_DEVISE          float64      

In [7]:
df_bt= pd.DataFrame(df_base_transpa)

df_bt.head()

,DATE_TRANSPA,DATE_VALEUR,SOURCE,GROUPE,CANTON,AFFECTATION,SECTION,CODE_CTN,CODE_PTF,LIB_PTF,...,CODE_NIV4,LIBELLE_P,ATTRIBUT_W_P,LIBELLE_W_P,OPC_P,ID_EMETTEUR_P,CLASSIF_SAA_P,CLASSIF_RF_P,SOUS_CLASSIF_RF_P,indic_tod
0,2024-12-31,2024-12-31,CHORUS,CGP,CGP RS,RS,,,5001,,...,AT0000383864,"AUTRICHE 6,25%97-27 S.6",OTF,Ligne à ligne taux fixe,0,529900QWWUI4XRVR7I03,Obligation Taux Fixe,Obligation,Souverain Taux Fixe,0
1,2024-12-31,2024-12-31,CHORUS,CGP,CGP AG,MDD,,,2006,,...,AT0000A04967,"AUTRICHE 4,15%07-150337",OTF,Ligne à ligne taux fixe,0,529900QWWUI4XRVR7I03,Obligation Taux Fixe,Obligation,Souverain Taux Fixe,0
2,2024-12-31,2024-12-31,CHORUS,CGP,CGP RS,RS,,,5001,,...,AT0000A04967,"AUTRICHE 4,15%07-150337",OTF,Ligne à ligne taux fixe,0,529900QWWUI4XRVR7I03,Obligation Taux Fixe,Obligation,Souverain Taux Fixe,0
3,2024-12-31,2024-12-31,CHORUS,CGP,CGP RS,RS,,,5001,,...,AT0000A0U299,"AUTRICHE 3,80%12-260162",OTF,Ligne à ligne taux fixe,0,529900QWWUI4XRVR7I03,Obligation Taux Fixe,Obligation,Souverain Taux Fixe,0
4,2024-12-31,2024-12-31,CHORUS,CGP,CGP AG,FP,,,3001,,...,AT0000A0VRQ6,"AUTRICHE 3,15%12-200644",OTF,Ligne à ligne taux fixe,0,529900QWWUI4XRVR7I03,Obligation Taux Fixe,Obligation,Souverain Taux Fixe,0


In [8]:
# Sauvegarde de df_base_transpa en CSV
# On s'assure que le dossier cleaned existe
os.makedirs("./cleaned/", exist_ok=True)

output_path = "./cleaned/base_transpa_eps.csv"
df_base_transpa.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./cleaned/base_transpa_eps.csv
Taille du fichier :  49.26 Mo


In [9]:
base_parent_path = "/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0/BP_SIM_EPS.json"

# vérifion d'abord que le fichier BASE_PARENT existe
p = Path(base_parent_path)
print("Fichier existe ?", p.exists())

if p.exists():
    df_base_parent = pd.read_json(p)
    print(df_base_parent.head())
else:
    raise FileNotFoundError(f"Fichier introuvable: {base_parent_path}")

Fichier existe ? True
  DATE_TRANSPA DATE_VALEUR   SOURCE         GROUPE    CANTON AFFECTATION  \
0    31DEC2024   31DEC2024  INTERNE  BPCE Mutuelle  BPCEM AG        MNCE   
1    31DEC2024   31DEC2024   CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
2    31DEC2024   31DEC2024   CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
3    31DEC2024   31DEC2024   CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
4    31DEC2024   31DEC2024   CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   

         SECTION  CODE_CTN  CODE_PTF      LIB_PTF  ... VM_PRE_TAUX_VAR99  \
0  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...      3.569423e+07   
1  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...      6.318023e+06   
2  BPCE MUTUELLE         8      8002   MNCE CELCA  ...      5.271688e+06   
3  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...      2.011153e+06   
4  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...      4.240076e+06   

   VM_PRE_CREDIT_VAR95 VM_PRE_CREDIT_VAR99 VM_PRE_ACTION_VAR95  

In [10]:
# Sauvegarde des données brutes de Base Parent
# On s'assure que le dossier cleaned existe
os.makedirs("./raw", exist_ok=True)

# Sauvegarde de df_base_parent en CSV
output_path = "./raw/raw_base_parent.csv"
df_base_parent.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")


Fichier sauvegardé avec succès : ./raw/raw_base_parent.csv
Taille du fichier :  25.04 Mo


In [11]:
df_bp = pd.DataFrame(df_base_parent)

df_bp.head()

,DATE_TRANSPA,DATE_VALEUR,SOURCE,GROUPE,CANTON,AFFECTATION,SECTION,CODE_CTN,CODE_PTF,LIB_PTF,...,VM_PRE_TAUX_VAR99,VM_PRE_CREDIT_VAR95,VM_PRE_CREDIT_VAR99,VM_PRE_ACTION_VAR95,VM_PRE_ACTION_VAR99,VM_PRE_IMMO_VAR95,VM_PRE_IMMO_VAR99,PROBA_PDD,SEVERITE_MOY_PDD,indic_tod
0,31DEC2024,31DEC2024,INTERNE,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,3.569423e+07,3.569423e+07,3.569423e+07,35694229.00,35694229.00,35694229.00,35694229.00,NaN,NaN,0
1,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,6.318023e+06,6.076382e+06,6.043657e+06,6128350.68,6128350.68,6128350.68,6128350.68,NaN,NaN,0
2,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8002,MNCE CELCA,...,5.271688e+06,5.072381e+06,5.045377e+06,5114369.86,5114369.86,5114369.86,5114369.86,NaN,NaN,0
3,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,2.011153e+06,2.006616e+06,2.006275e+06,2007138.25,2007138.25,2007138.25,2007138.25,NaN,NaN,0
4,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,4.240076e+06,4.038331e+06,4.018433e+06,4069093.15,4069093.15,4069093.15,4069093.15,NaN,NaN,0


In [12]:
# Sauvegarde de df_base_parent en CSV
# On s'assure que le dossier cleaned existe
os.makedirs("./cleaned", exist_ok=True)

output_path = "./cleaned/base_parent_eps.csv"
df_base_parent.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./cleaned/base_parent_eps.csv
Taille du fichier :  25.04 Mo


In [13]:
flux_passif_path = "/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0/PASSIF_SIM_EPS.json"

# vérifion d'abord que le fichier FLUX_PASSIF existe
p = Path(flux_passif_path)
print("Fichier existe ?", p.exists())

if p.exists():
    df_flux_passif = pd.read_json(p)
    print(df_flux_passif.head())
else:
    raise FileNotFoundError(f"Fichier introuvable: {flux_passif_path}")

Fichier existe ? True
  DATE_TRANSPA GROUPE  CANTON AFFECTATION LIBELLE TYPE_SIM       VM_INIT  \
0    31DEC2024    CGP  CGP AG         MDD  PASSIF     TAUX  4.469031e+09   
1    31DEC2024    CGP  CGP RS          RS  PASSIF     TAUX  2.135597e+09   
2    31MAR2025    CGP  CGP AG         MDD  PASSIF     TAUX  4.244303e+09   
3    31MAR2025    CGP  CGP RS          RS  PASSIF     TAUX  2.021071e+09   
4    30JUN2025    CGP  CGP AG         MDD  PASSIF     TAUX  4.245514e+09   

   VM_RC_GLOBAL_VAR95  VM_RC_GLOBAL_VAR99  VM_RC_TAUX_VAR95  ...       FLUX92  \
0        5.387582e+09        5.276745e+09      5.360841e+09  ...     0.000000   
1        2.596059e+09        2.588198e+09      2.573563e+09  ...  1336.739365   
2        5.272549e+09        4.992279e+09      4.819791e+09  ...     0.000000   
3        2.302988e+09        2.442847e+09      2.429030e+09  ...  1336.739365   
4        5.274227e+09        4.993825e+09      5.071296e+09  ...     0.000000   

       FLUX93     FLUX94      FLUX

In [14]:
# Sauvegarde des données brutes de Flux Passif
# On s'assure que le dossier cleaned existe
os.makedirs("./raw", exist_ok=True)

# Sauvegarde de df_flux_passif en CSV
output_path = "./raw/raw_passif.csv"
df_flux_passif.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")


Fichier sauvegardé avec succès : ./raw/raw_passif.csv
Taille du fichier :  17.02 Ko


In [15]:
df_flux_passif = pd.DataFrame(df_flux_passif)

df_flux_passif.head()

,DATE_TRANSPA,GROUPE,CANTON,AFFECTATION,LIBELLE,TYPE_SIM,VM_INIT,VM_RC_GLOBAL_VAR95,VM_RC_GLOBAL_VAR99,VM_RC_TAUX_VAR95,...,FLUX92,FLUX93,FLUX94,FLUX95,FLUX96,FLUX97,FLUX98,FLUX99,FLUX100,indic_tod
0,31DEC2024,CGP,CGP AG,MDD,PASSIF,TAUX,4.469031e+09,5.387582e+09,5.276745e+09,5.360841e+09,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0
1,31DEC2024,CGP,CGP RS,RS,PASSIF,TAUX,2.135597e+09,2.596059e+09,2.588198e+09,2.573563e+09,...,1336.739365,892.702929,615.47628,432.507976,299.755861,201.23324,128.929964,77.327224,41.861122,0
2,31MAR2025,CGP,CGP AG,MDD,PASSIF,TAUX,4.244303e+09,5.272549e+09,4.992279e+09,4.819791e+09,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0
3,31MAR2025,CGP,CGP RS,RS,PASSIF,TAUX,2.021071e+09,2.302988e+09,2.442847e+09,2.429030e+09,...,1336.739365,892.702929,615.47628,432.507976,299.755861,201.23324,128.929964,77.327224,41.861122,0
4,30JUN2025,CGP,CGP AG,MDD,PASSIF,TAUX,4.245514e+09,5.274227e+09,4.993825e+09,5.071296e+09,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0


In [16]:
# Sauvegarde de df_flux_passif en CSV
# On s'assure que le dossier cleaned existe
os.makedirs("./cleaned", exist_ok=True)

output_path = "./cleaned/passif_eps.csv"
df_flux_passif.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./cleaned/passif_eps.csv
Taille du fichier :  17.02 Ko


In [17]:
param_rrf_path = "/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0/PARAM_RRF.json"

# vérifion d'abord que le fichier PARAM_RRF existe
p = Path(param_rrf_path)
print("Fichier existe ?", p.exists())

if p.exists():
    df_param_rrf = pd.read_json(p)
    print(df_param_rrf.head())
else:
    raise FileNotFoundError(f"Fichier introuvable: {param_rrf_path}")


Fichier existe ? True
  DATE_TRANSPA            var  \
0    31DEC2024        REP_RRF   
1    31DEC2024        LIB_FMT   
2    31DEC2024     NB_TRANSPA   
3    31DEC2024  SEUIL_TRANSPA   
4    31DEC2024  FORCAGE_COURS   

                                                 val  indic_tod  
0  \\sv61file0024\Bureautique\Direction des Risqu...        NaN  
1  \\sv61file0024\Bureautique\Direction des Risqu...        NaN  
2                                                  4        NaN  
3                                             100000        NaN  
4                                                  1        NaN  


In [18]:
# Sauvegarde des données brutes de Param RFF
# On s'assure que le dossier cleaned existe
os.makedirs("./raw", exist_ok=True)

# Sauvegarde de df_param_rrf en CSV
output_path = "./raw/raw_param_rrf.csv"
df_param_rrf.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")


Fichier sauvegardé avec succès : ./raw/raw_param_rrf.csv
Taille du fichier :  5.46 Ko


In [19]:
df_param_rrf = pd.DataFrame(df_param_rrf)

df_param_rrf.head()

,DATE_TRANSPA,var,val,indic_tod
0,31DEC2024,REP_RRF,\\sv61file0024\Bureautique\Direction des Risqu...,NaN
1,31DEC2024,LIB_FMT,\\sv61file0024\Bureautique\Direction des Risqu...,NaN
2,31DEC2024,NB_TRANSPA,4,NaN
3,31DEC2024,SEUIL_TRANSPA,100000,NaN
4,31DEC2024,FORCAGE_COURS,1,NaN


In [20]:
# Sauvegarde des données brutes de PARAM_RRF
# On s'assure que le dossier cleaned existe
os.makedirs("./cleaned", exist_ok=True)

# Sauvegarde de df_param_rrf en CSV
output_path = "./cleaned/param_rrf.csv"

df_param_rrf.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./cleaned/param_rrf.csv
Taille du fichier :  5.46 Ko


In [21]:
va_sim_path = "/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0/VA_SIM_EPS.json"

# vérifion d'abord que le fichier BASE_PARENT existe
p = Path(va_sim_path)
print("Fichier existe ?", p.exists())

if p.exists():
    df_va_sim = pd.read_json(p)
    print(df_va_sim.head())
else:
    raise FileNotFoundError(f"Fichier introuvable: {va_sim_path}")


Fichier existe ? True
  DATE_TRANSPA DATE_VALEUR  SOURCE         GROUPE    CANTON AFFECTATION  \
0    31DEC2024   31DEC2024  CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
1    31DEC2024   31DEC2024  CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
2    31DEC2024   31DEC2024  CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
3    31DEC2024   31DEC2024  CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   
4    31DEC2024   31DEC2024  CHORUS  BPCE Mutuelle  BPCEM AG        MNCE   

         SECTION  CODE_CTN  CODE_PTF      LIB_PTF  ... FLUX52  FLUX53 FLUX54  \
0  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...    0.0     0.0    0.0   
1  BPCE MUTUELLE         8      8002   MNCE CELCA  ...    0.0     0.0    0.0   
2  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...    0.0     0.0    0.0   
3  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...    0.0     0.0    0.0   
4  BPCE MUTUELLE         8      8001  MNCE CACEIS  ...    0.0     0.0    0.0   

  FLUX55 FLUX56 FLUX57 FLUX58  FLUX59  FLUX60 

In [22]:
# Sauvegarde des données brutes du flux de Valeur Amortissable VA_FLUX
# On s'assure que le dossier cleaned existe
os.makedirs("./raw", exist_ok=True)

# Sauvegarde de df_va_sim en CSV
output_path = "./raw/raw_va_sim.csv"
df_va_sim.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./raw/raw_va_sim.csv
Taille du fichier :  12.47 Mo


In [23]:
df_va_sim = pd.DataFrame(df_va_sim)

df_va_sim.head()

,DATE_TRANSPA,DATE_VALEUR,SOURCE,GROUPE,CANTON,AFFECTATION,SECTION,CODE_CTN,CODE_PTF,LIB_PTF,...,FLUX52,FLUX53,FLUX54,FLUX55,FLUX56,FLUX57,FLUX58,FLUX59,FLUX60,indic_tod
0,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8002,MNCE CELCA,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,31DEC2024,31DEC2024,CHORUS,BPCE Mutuelle,BPCEM AG,MNCE,BPCE MUTUELLE,8,8001,MNCE CACEIS,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [24]:
# Sauvegarde des données cleaned du flux de Valeur Amortissable VA_FLUX
# On s'assure que le dossier cleaned existe
os.makedirs("./cleaned", exist_ok=True)

# Sauvegarde de df_va_sim en CSV
output_path = "./cleaned/va_sim.csv"
df_va_sim.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./cleaned/va_sim.csv
Taille du fichier :  12.47 Mo


In [25]:
indic_risque_path = "/mnt/risques/4. Risques Financiers/00-0-REPORTING/01 - HISTO SAS/0000T0/INDICATEURS_RISQUE.json"

# vérifion d'abord que le fichier INDICATEURS_RISQUE existe
p = Path(indic_risque_path)
print("Fichier existe ?", p.exists())

if p.exists():
    df_indic_risque = pd.read_json(p)
    print(df_indic_risque.head())
else:
    raise FileNotFoundError(f"Fichier introuvable: {indic_risque_path}")

Fichier existe ? True
  DATE_TRANSPA    CANTON INDICATEUR TYPE_SIM  VAR  Indic_Init     Indic  \
0    31DEC2024  BPCEM AG        PDD   ACTION   95    0.003523  0.010044   
1    31DEC2024  BPCEM AG        PDD   ACTION   99    0.003523  0.017119   
2    31DEC2024  BPCEM AG        PDD   CREDIT   95    0.003523  0.003523   
3    31DEC2024  BPCEM AG        PDD   CREDIT   99    0.003523  0.003523   
4    31DEC2024  BPCEM AG        PDD   GLOBAL   95    0.003523  0.009807   

   SEUIL1  SEUIL2      ASSIETTE  ...  Scenario  VAR_ZCN10Y_BP  VAR_ZCR10Y_BP  \
0  0.0025    0.02  1.702189e+08  ...     440.0            0.0         0.0000   
1  0.0025    0.02  1.702189e+08  ...     415.0            0.0         0.0000   
2  0.0025    0.02  1.702189e+08  ...      50.0            0.0         0.0000   
3  0.0025    0.02  1.702189e+08  ...      10.0            0.0         0.0000   
4  0.0025    0.02  1.702189e+08  ...     566.0           82.4        35.6092   

   VAR_BEI10Y_BP  VAR_SPD_BP   RDT_EQU  RDT_IM

In [26]:
# Sauvegarde des données brutes des Indicateurs Risque
# On s'assure que le dossier cleaned existe
os.makedirs("./raw", exist_ok=True)

# Sauvegarde de df_indic_risque en CSV
output_path = "./raw/raw_indicateurs_risque.csv"
df_indic_risque.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./raw/raw_indicateurs_risque.csv
Taille du fichier :  155.63 Ko


In [27]:
df_indic_risque = pd.DataFrame(df_indic_risque)

df_indic_risque.head()

,DATE_TRANSPA,CANTON,INDICATEUR,TYPE_SIM,VAR,Indic_Init,Indic,SEUIL1,SEUIL2,ASSIETTE,...,Scenario,VAR_ZCN10Y_BP,VAR_ZCR10Y_BP,VAR_BEI10Y_BP,VAR_SPD_BP,RDT_EQU,RDT_IMMO,DEFAUT_GOVT,DEFAUT_CORP,indic_tod
0,31DEC2024,BPCEM AG,PDD,ACTION,95,0.003523,0.010044,0.0025,0.02,1.702189e+08,...,440.0,0.0,0.0000,0.000000,0.0,-0.177958,0.000000,0.000000,0.000000,0
1,31DEC2024,BPCEM AG,PDD,ACTION,99,0.003523,0.017119,0.0025,0.02,1.702189e+08,...,415.0,0.0,0.0000,0.000000,0.0,-0.272477,0.000000,0.000000,0.000000,0
2,31DEC2024,BPCEM AG,PDD,CREDIT,95,0.003523,0.003523,0.0025,0.02,1.702189e+08,...,50.0,0.0,0.0000,0.000000,1.7,0.000000,0.000000,0.000000,0.000000,0
3,31DEC2024,BPCEM AG,PDD,CREDIT,99,0.003523,0.003523,0.0025,0.02,1.702189e+08,...,10.0,0.0,0.0000,0.000000,-3.9,0.000000,0.000000,0.000000,0.000000,0
4,31DEC2024,BPCEM AG,PDD,GLOBAL,95,0.003523,0.009807,0.0025,0.02,1.702189e+08,...,566.0,82.4,35.6092,45.686561,43.3,-0.188805,0.041164,0.001524,0.002134,0


In [28]:
# Catégories
cat_candidat = ['CANTON', 'INDICATEUR', 'TYPE_SIM', 'VAR', 'Scenario']
cat_cols = [ c for c in cat_candidat if c in df_indic_risque.columns]
if cat_cols :
    df_indic_risque[cat_cols] = df_indic_risque[cat_cols].astype('category')
    
# Dates :
df_indic_risque.columns = df_indic_risque.columns.str.strip()

date_candidat = ['DATE_TRANSPA']
date_cols = [ c for c in date_candidat if c in df_indic_risque.columns]

for col in date_cols:
    df_indic_risque[col] = pd.to_datetime(df_indic_risque[col], dayfirst=True, errors='coerce')
    
df_indic_risque.head()

/tmp/ipykernel_817281/160197258.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_indic_risque[col] = pd.to_datetime(df_indic_risque[col], dayfirst=True, errors='coerce')


,DATE_TRANSPA,CANTON,INDICATEUR,TYPE_SIM,VAR,Indic_Init,Indic,SEUIL1,SEUIL2,ASSIETTE,...,Scenario,VAR_ZCN10Y_BP,VAR_ZCR10Y_BP,VAR_BEI10Y_BP,VAR_SPD_BP,RDT_EQU,RDT_IMMO,DEFAUT_GOVT,DEFAUT_CORP,indic_tod
0,2024-12-31,BPCEM AG,PDD,ACTION,95,0.003523,0.010044,0.0025,0.02,1.702189e+08,...,440.0,0.0,0.0000,0.000000,0.0,-0.177958,0.000000,0.000000,0.000000,0
1,2024-12-31,BPCEM AG,PDD,ACTION,99,0.003523,0.017119,0.0025,0.02,1.702189e+08,...,415.0,0.0,0.0000,0.000000,0.0,-0.272477,0.000000,0.000000,0.000000,0
2,2024-12-31,BPCEM AG,PDD,CREDIT,95,0.003523,0.003523,0.0025,0.02,1.702189e+08,...,50.0,0.0,0.0000,0.000000,1.7,0.000000,0.000000,0.000000,0.000000,0
3,2024-12-31,BPCEM AG,PDD,CREDIT,99,0.003523,0.003523,0.0025,0.02,1.702189e+08,...,10.0,0.0,0.0000,0.000000,-3.9,0.000000,0.000000,0.000000,0.000000,0
4,2024-12-31,BPCEM AG,PDD,GLOBAL,95,0.003523,0.009807,0.0025,0.02,1.702189e+08,...,566.0,82.4,35.6092,45.686561,43.3,-0.188805,0.041164,0.001524,0.002134,0


In [29]:
# Sauvegarde des données cleaned des indicateurs de risque
# On s'assure que le dossier cleaned existe
os.makedirs("./cleaned", exist_ok=True)

# Sauvegarde de df_indic_risque en CSV
output_path = "./cleaned/indic_risque.csv"
df_indic_risque.to_csv(output_path, index=False, encoding="utf-8-sig")

# Vérification de la sauvegarde
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
# en octects

    # Conversion en Ko ou en Mo
    if  file_size < 1024:
        size_str = f"{file_size} octects"
    elif file_size < 1024**2:
        size_str = f"{file_size/1024: .2f} Ko"
    else :
        size_str = f"{file_size/1024**2: .2f} Mo"
    
    print(f"Fichier sauvegardé avec succès : {output_path}")
    print(f"Taille du fichier : {size_str}")

else:
    print(f"Erreur :le fichier n'a pas été créé dans  {output_path}")

Fichier sauvegardé avec succès : ./cleaned/indic_risque.csv
Taille du fichier :  156.68 Ko
